# Exercise 21.1: A simple calcium cycling model

We will implement a simple calcium cycling model based on sympathetic neurons (Friel et al., 1995). The model tracks only two states: the cytosolic calcium $[\mathrm{Ca}^{2+}]_{\mathrm{i}}$ and the SR calcium $[\mathrm{Ca}^{2+}]_{\mathrm{SR}}$.

There are four fluxes:

1. $J_{\mathrm{entry}}$: Calcium entering the cell from outside.
2. $J_{\mathrm{extrusion}}$: Calcium pumped out of the cell.
3. $J_{\mathrm{rel}}$: Calcium released from the SR (CICR).
4. $J_{\mathrm{uptake}}$: Calcium pumped into the SR.

The rates of change are determined by these fluxes:

$$
\frac{\mathrm{d}[\mathrm{Ca}^{2+}]_{\mathrm{i}}}{\mathrm{d}t} = J_{\mathrm{entry}} - J_{\mathrm{extrusion}} + J_{\mathrm{rel}} - J_{\mathrm{uptake}}
$$

$$
\frac{\mathrm{d}[\mathrm{Ca}^{2+}]_{\mathrm{SR}}}{\mathrm{d}t} = \frac{J_{\mathrm{uptake}} - J_{\mathrm{rel}}}{\gamma}
$$

_(Note: $\gamma = 0.24$ is a volume scaling factor. Because the SR is physically much smaller than the cytosol, moving 1 ion into the SR spikes its concentration much faster than moving 1 ion into the vast cytosol)._


## Exercise 21.1a: Implementing the RHS

To generate CICR oscillations, the release rate from the SR must be highly sensitive to the cytosolic calcium. We model the rate constant $k_{\mathrm{rel}}$ as a steep Hill equation:

$$
k_{\mathrm{rel}} = \kappa_0 + \kappa_1 \frac{[\mathrm{Ca}^{2+}]_{\mathrm{i}}^n}{K_{\mathrm{d}}^n + [\mathrm{Ca}^{2+}]_{\mathrm{i}}^n}
$$

Complete the Python code below to define the fluxes and the derivatives.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


def rhs_friel(t, y, Cao, k_entry, k_extrusion, k_uptake, kappa0, kappa1, Kd, n, gamma):
    # Unpack the state variables
    Cai, CaSR = y

    # Define the linear fluxes
    J_entry = k_entry * (Cao - Cai)
    J_extrusion = k_extrusion * Cai
    J_uptake = k_uptake * Cai

    # Compute the non-linear release rate (CICR)
    k_rel = kappa0 + kappa1 * (Cai**n) / (Kd**n + Cai**n)
    J_rel = k_rel * (CaSR - Cai)

    # Calculate the derivatives
    dCai_dt = J_entry - J_extrusion + J_rel - J_uptake
    dCaSR_dt = (J_uptake - J_rel) / gamma

    return [dCai_dt, dCaSR_dt]

## Exercise 21.1b: Solving and Plotting

Use `solve_ivp` to simulate the model for 1000 seconds. Initial conditions are 80 nM (0.08 µM) for the cytosol and 4.0 µM for the SR.


In [ ]:
# Parameters
Cao = 1000.0  # µM
k_entry = 2e-5  # 1/s
k_extrusion = 0.132  # 1/s
k_uptake = 0.9  # 1/s
kappa0 = 0.013  # 1/s
kappa1 = 0.58  # 1/s
Kd = 0.5  # µM
n = 3.0
gamma = 0.24

params = (Cao, k_entry, k_extrusion, k_uptake, kappa0, kappa1, Kd, n, gamma)
y0 = [0.08, 4.0]
t_span = (0, 1000)

sol = solve_ivp(...)

# Plot the cytosolic calcium over time
plt.plot(sol.t, sol.y[0], label="Cytosolic Ca2+")
plt.ylabel("[Ca2+]i (µM)")
plt.show()

# Plot the SR calcium over time
plt.plot(...)